# Doanh thu + thói quen khách hàng theo khu vực & độ tuổi

Kết hợp `order_items` + `orders` + `products` + `customers` + `geography`. Công thức revenue = gross
(`qty*unit_price`, mọi trạng thái đơn) — khớp `sales.csv` 0.000% sai lệch (Phase 0 + `EDA_diagnostic.ipynb`).

Chạy từng cell bằng **Shift+Enter**, kernel `.venv`. Cell đầu load + join, dùng lại `li` cho cả bài.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

C_REV, C_COGS, C_GRAY, C_ACCENT = "#A32D2D", "#185FA5", "#888780", "#534AB7"
PALETTE = ["#185FA5", "#A32D2D", "#3B6D11", "#BA7517", "#534AB7", "#993556"]
AGE_ORDER = ["18-24", "25-34", "35-44", "45-54", "55+"]

plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "grid.linestyle": ":"})

DATA = Path("../../data")
orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA / "order_items.csv")
products = pd.read_csv(DATA / "products.csv", usecols=["product_id", "category", "cogs"])
customers = pd.read_csv(DATA / "customers.csv", usecols=["customer_id", "age_group", "gender", "signup_date"])
geography = pd.read_csv(DATA / "geography.csv", usecols=["zip", "region"])

li = (items
      .merge(orders, on="order_id", how="left")
      .merge(products, on="product_id", how="left")
      .merge(customers, on="customer_id", how="left")
      .merge(geography, on="zip", how="left"))

li["line_revenue"] = li["quantity"] * li["unit_price"]
li["year"] = li["order_date"].dt.year
li = li[li["year"].between(2013, 2022)].copy()
li["age_group"] = pd.Categorical(li["age_group"], categories=AGE_ORDER, ordered=True)

orders_per_cust = orders.groupby("customer_id").size().rename("n_orders")

print(f"{len(li):,} dòng sau join | {li['customer_id'].nunique():,} khách | age_group null: {li['age_group'].isna().sum()}")

## 1. Doanh thu theo nhóm tuổi qua các năm

Tỷ trọng gần như đứng yên suốt 10 năm (mỗi nhóm tuổi lệch <1 điểm % giữa 2013 và 2022) — cơ cấu tuổi khách hàng không dịch chuyển.

In [ ]:
age_yr = li.groupby(["year", "age_group"], observed=True)["line_revenue"].sum().unstack()
age_share = age_yr.div(age_yr.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5.5))
bottom = np.zeros(len(age_share))
for i, ag in enumerate(AGE_ORDER):
    ax.bar(age_share.index, age_share[ag], bottom=bottom, label=ag,
           color=PALETTE[i % len(PALETTE)], width=0.65)
    bottom += age_share[ag].values
ax.set_ylabel("% doanh thu trong năm")
ax.set_title("Tỷ trọng doanh thu theo nhóm tuổi qua các năm")
ax.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.08))
plt.show()

print(f"2013: {age_share.loc[2013].round(1).to_dict()}")
print(f"2022: {age_share.loc[2022].round(1).to_dict()}")

## 2. Heatmap doanh thu: độ tuổi × khu vực (tổng 2013–2022)

**Đọc cẩn thận** — đây là TỔNG, bị chi phối bởi số lượng khách mỗi ô. East luôn đậm nhất không có nghĩa khách East "chịu chi" hơn — xem cell per-capita ngay sau để tách 2 yếu tố này.

In [ ]:
cross = li.pivot_table(index="age_group", columns="region", values="line_revenue",
                       aggfunc="sum", observed=True).reindex(AGE_ORDER) / 1e9

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(cross.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(cross.columns)), cross.columns)
ax.set_yticks(range(len(cross.index)), cross.index)
for i in range(cross.shape[0]):
    for j in range(cross.shape[1]):
        v = cross.values[i, j]
        color = "white" if v > cross.values.max() * 0.6 else "#2c2c2a"
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", color=color, fontsize=10)
ax.grid(False)
fig.colorbar(im, label="Revenue (tỷ VND)")
ax.set_title("Doanh thu: độ tuổi × khu vực (tỷ VND, 2013–2022)")
plt.show()

print(cross.round(2).to_string())

## 3. AOV theo nhóm tuổi

25.42k–25.59k VND — chênh lệch <1%. Tuổi không ảnh hưởng giá trị đơn hàng.

In [ ]:
order_val = li.groupby(["order_id", "age_group"], observed=True)["line_revenue"].sum().reset_index()
aov_age = order_val.groupby("age_group", observed=True)["line_revenue"].mean().reindex(AGE_ORDER)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(aov_age.index, aov_age.values / 1e3, color=C_ACCENT, width=0.55)
for i, v in enumerate(aov_age.values / 1e3):
    ax.text(i, v, f"{v:.1f}k", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Giá trị đơn TB (nghìn VND)")
ax.set_title("AOV theo nhóm tuổi")
plt.show()

print(f"AOV (nghìn VND): {(aov_age/1e3).round(2).to_dict()}")

## 4. Tần suất mua theo nhóm tuổi

7.07–7.27 đơn/khách suốt vòng đời trong data — cũng gần như phẳng.

In [ ]:
cust_age = customers.set_index("customer_id")["age_group"]
freq = orders_per_cust.to_frame().join(cust_age).dropna()
freq_age = freq.groupby("age_group", observed=True)["n_orders"].mean().reindex(AGE_ORDER)
n_cust_age = freq.groupby("age_group", observed=True).size().reindex(AGE_ORDER)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(freq_age.index, freq_age.values, color=C_COGS, width=0.55)
for i, v in enumerate(freq_age.values):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Số đơn TB / khách")
ax.set_title("Tần suất mua theo nhóm tuổi")
plt.show()

print(f"đơn/khách: {freq_age.round(2).to_dict()}")
print(f"số khách/nhóm: {n_cust_age.to_dict()}")

## 5. Category ưa thích theo nhóm tuổi

Streetwear ~80% ở MỌI nhóm tuổi, Outdoor ~15% ở mọi nhóm — sở thích category không đổi theo tuổi, khác biệt category (chart Diagnostic trước) là theo NĂM chứ không theo ai mua.

In [ ]:
cat_age = li.pivot_table(index="age_group", columns="category", values="line_revenue",
                         aggfunc="sum", observed=True).reindex(AGE_ORDER)
cat_age_share = cat_age.div(cat_age.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cat_age_share.values, cmap="Oranges", aspect="auto")
ax.set_xticks(range(len(cat_age_share.columns)), cat_age_share.columns)
ax.set_yticks(range(len(cat_age_share.index)), cat_age_share.index)
for i in range(cat_age_share.shape[0]):
    for j in range(cat_age_share.shape[1]):
        v = cat_age_share.values[i, j]
        color = "white" if v > 50 else "#2c2c2a"
        ax.text(j, i, f"{v:.1f}%", ha="center", va="center", color=color, fontsize=10)
ax.grid(False)
fig.colorbar(im, label="% doanh thu trong nhóm tuổi")
ax.set_title("Category ưa thích theo nhóm tuổi (% trong từng hàng)")
plt.show()

print(cat_age_share.round(1).to_string())

## 6. Giới tính — revenue, AOV, tần suất

Revenue tỷ lệ đúng theo số lượng khách mỗi giới (Female 59,640 khách vs Male 57,457) — AOV và tần suất mua gần như bằng nhau giữa các giới.

In [ ]:
gender_rev = li.groupby("gender")["line_revenue"].sum() / 1e9
order_val_g = li.groupby(["order_id", "gender"])["line_revenue"].sum().reset_index()
aov_gender = order_val_g.groupby("gender")["line_revenue"].mean()
cust_gender = customers.set_index("customer_id")["gender"]
freq_g = orders_per_cust.to_frame().join(cust_gender).dropna().groupby("gender")["n_orders"].mean()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].bar(gender_rev.index, gender_rev.values, color=PALETTE[:3], width=0.55)
axes[0].set_ylabel("Revenue (tỷ VND)"); axes[0].set_title("Doanh thu theo giới tính")
axes[1].bar(aov_gender.index, aov_gender.values / 1e3, color=PALETTE[:3], width=0.55)
axes[1].set_ylabel("AOV (nghìn VND)"); axes[1].set_title("AOV theo giới tính")
axes[2].bar(freq_g.index, freq_g.values, color=PALETTE[:3], width=0.55)
axes[2].set_ylabel("Đơn TB/khách"); axes[2].set_title("Tần suất mua theo giới tính")
plt.show()

print(f"Revenue (tỷ): {gender_rev.round(2).to_dict()}")
print(f"AOV (nghìn): {(aov_gender/1e3).round(2).to_dict()}")
print(f"Tần suất: {freq_g.round(2).to_dict()}")

## 7. Revenue / khách (per-capita) — chart quan trọng nhất bài này

Tách ảnh hưởng "số lượng khách" ra khỏi "hành vi thật":
- **Theo tuổi**: gần như phẳng (0.126–0.131 triệu/khách) — tuổi không phải yếu tố phân hoá.
- **Theo khu vực**: KHÔNG phẳng — khách **West chi tiêu/người cao hơn Central ~62%**, dù West có ít khách nhất (14,741 so với East 44,721). Nhìn tổng (chart 2) sẽ tưởng West "kém", per-capita thì ngược lại. Đúng theo mọi nhóm tuổi (không phải trùng hợp do lệch cơ cấu tuổi).

In [ ]:
cust_rev = li.groupby("customer_id")["line_revenue"].sum().rename("total_rev")
cust_dim = customers.set_index("customer_id")[["age_group", "gender"]].join(
    orders[["customer_id", "zip"]].drop_duplicates("customer_id").set_index("customer_id")
).join(geography.set_index("zip")["region"], on="zip")
percap = cust_dim.join(cust_rev).fillna({"total_rev": 0})

percap_region = percap.groupby("region")["total_rev"].mean() / 1e6
percap_age = percap.groupby("age_group", observed=True)["total_rev"].mean().reindex(AGE_ORDER) / 1e6
n_region = percap["region"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for i, (reg, v) in enumerate(percap_region.items()):
    axes[0].bar(reg, v, color=PALETTE[i % len(PALETTE)], width=0.55)
    axes[0].text(i, v, f"{v:.3f}tr\n({n_region[reg]:,} khách)", ha="center", va="bottom", fontsize=9)
axes[0].set_ylabel("Revenue TB / khách (triệu VND)")
axes[0].set_title("Per-capita theo khu vực — KHÔNG phẳng")

axes[1].bar(percap_age.index, percap_age.values, color=C_GRAY, width=0.55)
for i, v in enumerate(percap_age.values):
    axes[1].text(i, v, f"{v:.3f}tr", ha="center", va="bottom", fontsize=9)
axes[1].set_ylabel("Revenue TB / khách (triệu VND)")
axes[1].set_title("Per-capita theo tuổi — gần như phẳng")
ymax = percap_region.max() * 1.15
axes[0].set_ylim(0, ymax); axes[1].set_ylim(0, ymax)
plt.show()

print(f"Theo region: {percap_region.round(4).to_dict()}")
print(f"Theo tuổi  : {percap_age.round(4).to_dict()}")
west_vs_central = (percap_region["West"] / percap_region["Central"] - 1) * 100
print(f"West cao hơn Central: {west_vs_central:.1f}%")

## Chốt insight

| Câu hỏi | Trả lời |
|---|---|
| Tuổi có ảnh hưởng doanh thu/khách, AOV, tần suất, category? | **Không** — mọi chỉ số gần như phẳng qua 5 nhóm tuổi |
| Khu vực có ảnh hưởng? | **Có** — West chi/người cao hơn Central 62%, dù dân số khách ít nhất |
| Giới tính có ảnh hưởng? | Không — AOV/tần suất bằng nhau, revenue tổng chỉ theo tỷ lệ dân số |
| Bẫy cần tránh | Đọc chart TỔNG theo khu vực mà không đối chiếu per-capita — dễ kết luận ngược |

**Prescriptive gợi ý**: nếu cân nhắc mở rộng theo vùng, West đáng đầu tư hơn per-customer dù quy mô đang nhỏ nhất — ưu tiên khác với việc chỉ nhìn "vùng nào đang mang doanh thu nhiều nhất" (East). Age_group không đáng dùng làm biến phân khúc — dữ liệu không ủng hộ.